# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [ ]:
%pip install python-dotenv

%pip install jupyter_bokeh

%pip install panel

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

print(OPENAI_API_KEY is not None)  # should print True

True


In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [ ]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [ ]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [3]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [ ]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages



inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [4]:
import panel as pn
pn.extension()  # load Panel's JavaScript in its own cell so the chat windows render

In [5]:
def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def make_bot(system_prompt):
    context = [{'role': 'system', 'content': system_prompt}]
    panels = []
    inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
    button = pn.widgets.Button(name="Chat!")

    def collect(_):
        prompt = inp.value_input
        if not prompt:
            return pn.Column(*panels)
        inp.value = ''
        context.append({'role': 'user', 'content': prompt})
        response = get_completion_from_messages(context)
        context.append({'role': 'assistant', 'content': response})
        panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
        panels.append(pn.Row('Assistant:', pn.pane.Markdown(
            response, width=600, styles={'background-color': '#F6F6F6', 'color': 'black'})))
        return pn.Column(*panels)

    dashboard = pn.Column(
        inp,
        pn.Row(button),
        pn.panel(pn.bind(collect, button), loading_indicator=True, height=300),
    )
    return dashboard, context


def order_summary(context):
    if not any(m['role'] == 'user' for m in context):
        print('No order yet. Chat with the bot first, then run this cell.')
        return
    messages = context.copy()
    messages.append({'role': 'system', 'content':
        'create a json summary of the previous food order. Itemize the price for each item. '
        'The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size '
        '4) list of sides include size 5) total price'})
    print(get_completion_from_messages(messages, temperature=0))

### Version 1: Original prompt (baseline)
The menu lists prices like `12.95, 10.00, 7.00` without saying which size is which. Watch whether the bot asks or guesses.

In [6]:
version_1 = """
You are OrderBot, an automated service to collect orders for a pizza restaurant.
You first greet the customer, then collect the order,
and then ask if it's a pickup or delivery.
You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else.
If it's a delivery, you ask for an address.
Finally you collect the payment.
Make sure to clarify all options, extras and sizes to uniquely
identify the item from the menu.
You respond in a short, very conversational friendly style.
The menu includes
pepperoni pizza  12.95, 10.00, 7.00
cheese pizza   10.95, 9.25, 6.50
eggplant pizza   11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
"""

dashboard_1, context_1 = make_bot(version_1)
dashboard_1

BokehModel(combine_events=True, render_bundle={'docs_json': {'dcaec6e1-88fc-4827-a652-0a9a53a915a9': {'version…

In [ ]:
order_summary(context_1)

### Version 2: Stricter, clearer prompt
Same restaurant, but sizes are labeled and there are explicit rules about off-menu items, unknown information and totals.

In [ ]:
version_2 = """
You are OrderBot, an automated service to collect orders for a pizza restaurant.

Steps:
1. Greet the customer.
2. Collect the full order. Always ask for the size if the customer didn't give one. Never assume a size.
3. Ask if it's pickup or delivery. If delivery, ask for the address.
4. Summarize the order with the price of each item, then calculate the total step by step.
5. Ask if they want anything else.
6. Ask for the payment method (cash or card).

Rules:
- Only offer items and toppings that are on the menu. If asked for something else, say it's not available and suggest something from the menu.
- If the customer asks something the menu doesn't answer (ingredients, allergens, delivery time, discounts), say you don't have that information. Never make it up.
- Never give discounts or free items.
- Stay on topic: only help with food orders.
- Respond in a short, friendly, conversational style.

Menu (prices in USD):
Pizzas (large / medium / small):
- pepperoni pizza: 12.95 / 10.00 / 7.00
- cheese pizza: 10.95 / 9.25 / 6.50
- eggplant pizza: 11.95 / 9.75 / 6.75
Sides:
- fries: large 4.50 / small 3.50
- greek salad: 7.25
Toppings (per pizza):
- extra cheese 2.00, mushrooms 1.50, sausage 3.00, canadian bacon 3.50, AI sauce 1.50, peppers 1.00
Drinks (large / medium / small):
- coke: 3.00 / 2.00 / 1.00
- sprite: 3.00 / 2.00 / 1.00
- bottled water: 5.00 (one size)
"""

dashboard_2, context_2 = make_bot(version_2)
dashboard_2

In [ ]:
order_summary(context_2)

### Version 3: Creative, a Berlin döner shop
A different restaurant and personality. Allergen info is given for some items but not all, to test whether the bot invents answers for the rest.

The JSON summary still asks for pizza fields, so expect it to look odd here. That mismatch is itself a finding.

In [ ]:
version_3 = """
You are DönerBot, the order assistant for "Kreuzberg Kebab", a small döner shop in Berlin.
You speak English but sprinkle in friendly Berlin slang now and then (e.g. "Na?", "Jut!").
Keep answers short and fun.

Flow: greet, take the order, ask "hier essen oder mitnehmen?" (eat in or take away),
summarize the order with prices, calculate the total step by step, then ask cash or card.

Rules:
- Only sell what is on the menu. Politely refuse anything else.
- Always ask which sauce if the customer doesn't say.
- Allergen info is listed below. If an item has no allergen info, say you don't know and suggest asking staff. Never guess.

Menu (EUR):
- Döner Kebab (bread): 7.50  | allergens: gluten, sesame
- Dürüm (wrap): 8.00         | allergens: gluten
- Döner Box with fries: 8.50
- Falafel Dürüm (vegan): 7.50 | allergens: gluten, sesame
- Lahmacun: 5.00
Sauces (free): garlic, herb, spicy
Extras: cheese 1.00, extra meat 2.50, halloumi 2.00
Drinks: Ayran 2.00, Club-Mate 3.00, Cola 2.50
"""

dashboard_3, context_3 = make_bot(version_3)
dashboard_3

In [ ]:
order_summary(context_3)

### Notes for the report

**Version 1 (original):**
- What worked: Once the setup was fixed, the bot followed the basic structure (greeting, order, pickup/delivery, payment) and stayed friendly and short.
- Errors / hallucinations:
Wrong total: for a large pepperoni with extra cheese and mushrooms plus a medium coke, the JSON summary gave 17.45. The correct total is 12.95 + 2.00 + 1.50 + 2.00 = 18.45.
Size confusion: in another summary, the pizza’s “size” was listed as “12.95”, a price instead of a size. The menu never says which price is large, medium or small, so the model has to guess.
Ignored requests: when I asked for pineapple, the bot repeated the same summary and didn’t respond to the request at all. This was caused by a bug in the helper function, which sent the wrong message list, so the model never saw my message.

**Version 2 (strict):**
- What worked: Labeled sizes should remove the size guessing, and the explicit rules should make it refuse off-menu items like pineapple and say “I don’t know” about allergens.
- Errors / hallucinations: Totals can still be off on long orders even with “calculate step by step”. The stricter prompt can also make the bot more rigid and repetitive.
- Did it fix version 1's problems? Mostly the size confusion, and probably not the math entirely.

**Version 3 (döner shop):**
- What worked: The personality and Berlin slang should come through well, and it should ask about sauce.
- Errors / hallucinations: The interesting test is items without allergen info, such as the Lahmacun. The model may still invent an answer despite the rule. The JSON summary also asks for pizza fields that don’t fit a döner menu, so its output gets forced into the wrong structure.

**What I learned:**
- The model has no memory of its own. It only knows the messages sent with each request. The bug in my notebook showed this clearly: when the helper sent the wrong list, the bot ignored everything I typed.
- When information is missing or ambiguous, like unlabeled prices, the model fills the gap with a confident guess instead of asking.
- Arithmetic is unreliable. The model produced a wrong total that looked completely plausible, so any numbers it gives should be checked or calculated in code.
- Clear structure in the prompt (labeled sizes, numbered steps, explicit “never make things up” rules) reduces errors but doesn’t eliminate them.
- The system prompt shapes the bot’s personality and behavior a lot, but the same prompt can’t be reused for a different task without adapting, as the pizza-shaped JSON for a döner order shows.